# HyperParameter Tunning

In [1]:
# Import

import os
import sys
import pandas as pd
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform , randint
from sklearn.metrics import f1_score

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath(".."))

from preprocessing import get_X_y

In [2]:
# Data Load

df_train = pd.read_csv(r"../dataset/UNSW_NB15_training-set.csv")

df_train_random = df_train.sample(frac = 0.4)

X_train , y_train = get_X_y(df_train_random)

df_test = pd.read_csv(r"../dataset/UNSW_NB15_testing-set.csv")

X_test , y_test = get_X_y(df_test)

In [4]:
# 1st Try

cat_cols = X_train.select_dtypes(include = ["object" , "string"]).columns
num_cols = X_train.select_dtypes(include = ["number"]).columns

weight = compute_sample_weight(
    y = y_train , 
    class_weight = "balanced"
)

preprocess = ColumnTransformer([
    ("cat" , OrdinalEncoder(handle_unknown = "use_encoded_value" , unknown_value = -1) , cat_cols) ,
    ("num" , "passthrough" , num_cols)
])

model = Pipeline([
    ("preprocess" , preprocess) , 
    ("xgbc" , XGBClassifier(
        n_jobs = 1 ,
        subsample = 0.8 , 
        colsample_bytree = 0.8 ,
        random_state = 42
    ))
])

param = {
    "xgbc__n_estimators" : randint(200 , 401) , 
    "xgbc__max_depth" : randint(4 , 9) ,
    "xgbc__learning_rate" : uniform(0.01 , 0.29) ,
    "xgbc__gamma" : uniform(0 , 5) ,
    "xgbc__reg_lambda" : uniform(0 , 30) ,
    "xgbc__reg_alpha" : uniform(0 , 10) ,
    "xgbc__min_child_weight" : [1,2,3,4,5]
    
}

srch = RandomizedSearchCV(
    estimator = model , 
    param_distributions = param ,
    cv = 3 ,
    n_iter = 10 ,
    n_jobs = -1
)

srch.fit(
    X_train ,
    y_train , 
    xgbc__sample_weight = weight
)

model = srch.best_estimator_

y_pred = model.predict(X_test)


print(srch.best_params_)
print()

print(f1_score(y_test , y_pred , average = "weighted"))

{'xgbc__gamma': np.float64(1.5722497595947744), 'xgbc__learning_rate': np.float64(0.25965777260613815), 'xgbc__max_depth': 8, 'xgbc__min_child_weight': 4, 'xgbc__n_estimators': 365, 'xgbc__reg_alpha': np.float64(2.7667692017035272), 'xgbc__reg_lambda': np.float64(8.566515158201373)}

0.7321187028540752


# Considering max_depth = 8  &  n_estimators = 365  &  min_child_weight = 4 as base line

In [8]:
# 2nd Try

cat_cols = X_train.select_dtypes(include = ["object" , "string"]).columns
num_cols = X_train.select_dtypes(include = ["number"]).columns

weight = compute_sample_weight(
    y = y_train , 
    class_weight = "balanced"
)

preprocess = ColumnTransformer([
    ("cat" , OrdinalEncoder(handle_unknown = "use_encoded_value" , unknown_value = -1) , cat_cols) ,
    ("num" , "passthrough" , num_cols)
])

model = Pipeline([
    ("preprocess" , preprocess) , 
    ("xgbc" , XGBClassifier(
        n_jobs = 1 ,
        subsample = 0.8 , 
        colsample_bytree = 0.8 ,
        random_state = 42 ,
        n_estimators = 365 ,
        min_child_weight = 4 ,
        max_depth = 8
    ))
])

param = {
    "xgbc__learning_rate" : uniform(0.20 , 0.20) ,
    "xgbc__gamma" : uniform(1 , 2) ,
    "xgbc__reg_lambda" : uniform(5 , 10) ,
    "xgbc__reg_alpha" : uniform(0 , 5) ,  
}

srch = RandomizedSearchCV(
    estimator = model , 
    param_distributions = param ,
    cv = 3 ,
    n_iter = 10 ,
    n_jobs = -1
)

srch.fit(
    X_train ,
    y_train , 
    xgbc__sample_weight = weight
)

model = srch.best_estimator_

y_pred = model.predict(X_test)


print(srch.best_params_)
print()

print(f1_score(y_test , y_pred , average = "weighted"))

{'xgbc__gamma': np.float64(1.1979048674817403), 'xgbc__learning_rate': np.float64(0.35224127984163117), 'xgbc__reg_alpha': np.float64(3.012854065663717), 'xgbc__reg_lambda': np.float64(12.515291566312271)}

0.7336980744501457


# Final Verdict
#-------------------------------------------------------------
# max_depth = 8 
# min_child_weight = 4 
# n_estimators = 365
# gamma = 1.1979
# reg_lambda = 12.5152
# reg_alpha = 3.0128
# learning_rate = 0.3522